# Tema Programare: Strategii Avansate de Validare

Acest notebook colecteaza fluxurile de validare mai specializate: impartire cronologica train/test, validare walk-forward, cross-validare nested si implementari de la zero ale logicii de impartire K-fold si stratified K-fold.


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau atunci cand este mentionat explicit ca poti interactiona cu ele.
* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul solutiei. Nu adauga si nu modifica cod in afara acestor comentarii.
* Poti adauga celule suplimentare pentru experimente, dar evaluatorul va verifica doar functiile evaluate.
* Evita folosirea variabilelor globale, cu exceptia cazului in care o celula iti spune explicit sa faci asta.
* Salveaza notebook-ul inainte de a-l trimite.
---


## Cuprins
- [Importuri](#imports)
- [1 - Impartire sensibila la timp](#1)
    - **[Exercitiul 1 - create_time_split](#ex-1)**
    - **[Exercitiul 2 - create_walk_forward_splits](#ex-2)**
- [2 - Cross-validare nested](#2)
    - **[Exercitiul 3 - run_nested_cv](#ex-3)**
    - **[Exercitiul 4 - compare_nested_vs_non_nested](#ex-4)**
- [3 - Cross-validare de la zero](#3)
    - **[Exercitiul 5 - kfold_indices](#ex-5)**
    - **[Exercitiul 6 - stratified_kfold_indices](#ex-6)**


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import unittests

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


<a name='1'></a>
## 1 - Impartire sensibila la timp


In [ ]:
rng = np.random.default_rng(42)
dates = pd.date_range('2021-01-01', periods=240, freq='D')
base_signal = np.sin(np.arange(240) / 12.0) + np.arange(240) * 0.01 + rng.normal(0, 0.1, 240)
time_df = pd.DataFrame(
    {
        'date': dates,
        'lag_1': pd.Series(base_signal).shift(1).bfill(),
        'lag_7': pd.Series(base_signal).shift(7).bfill(),
        'target': base_signal,
    }
)
time_df.head()


<a name='ex-1'></a>
### Exercitiul 1 - `create_time_split`

Sorteaza datele dupa timp, imparte-le fara amestecare si returneaza un dictionar cu seturile de train si test in ordine cronologica.


In [ ]:
# GRADED FUNCTION: create_time_split
def create_time_split(df, target_col='target', date_col='date', test_size=0.2):
    """
    Create a chronological train/test split for temporal data.
    """
    ### ÎNCEPUT DE COD AICI ### (approximately 7 lines)
    result = None
    ### SFÂRȘIT DE COD AICI ###
    return result


In [ ]:
time_split = create_time_split(time_df)
for key, value in time_split.items():
    print(key, value.shape)


<a name='ex-2'></a>
### Exercitiul 2 - `create_walk_forward_splits`

Implementeaza split-uri simple cu fereastra in expansiune. Fiecare split trebuie sa antreneze pe trecut si sa valideze pe un orizont viitor.


In [ ]:
# GRADED FUNCTION: create_walk_forward_splits
def create_walk_forward_splits(n_samples, initial_window, horizon, step=1):
    """
    Build expanding-window train/test index splits.
    """
    ### ÎNCEPUT DE COD AICI ### (approximately 9 lines)
    splits = None
    ### SFÂRȘIT DE COD AICI ###
    return splits


In [ ]:
walk_forward = create_walk_forward_splits(n_samples=60, initial_window=30, horizon=10, step=10)
walk_forward[:2]


In [ ]:
unittests.exercise_1(create_time_split)
unittests.exercise_2(create_walk_forward_splits)


<a name='2'></a>
## 2 - Cross-validare nested


In [ ]:
X_nested, y_nested = load_breast_cancer(return_X_y=True, as_frame=True)
X_nested = X_nested.iloc[:250].copy()
y_nested = y_nested.iloc[:250].copy()
base_model = LogisticRegression(max_iter=2000, solver='liblinear')
param_grid = {'C': [0.1, 1.0, 10.0]}


<a name='ex-3'></a>
### Exercitiul 3 - `run_nested_cv`

Pentru fiecare fold exterior, ruleaza un `GridSearchCV` interior, selecteaza cei mai buni hiperparametri si evalueaza modelul selectat pe fold-ul exterior lasat deoparte.


In [ ]:
# GRADED FUNCTION: run_nested_cv
def run_nested_cv(model, param_grid, X, y, outer_splits=3, inner_splits=2, scoring='accuracy', random_state=42):
    """
    Run nested cross-validation and return outer-fold evaluation results.
    """
    ### ÎNCEPUT DE COD AICI ### (approximately 18 lines)
    result = None
    ### SFÂRȘIT DE COD AICI ###
    return result


In [ ]:
nested_results = run_nested_cv(base_model, param_grid, X_nested, y_nested)
nested_results


<a name='ex-4'></a>
### Exercitiul 4 - `compare_nested_vs_non_nested`

Compara estimarea nested cu un scor de tuning non-nested pentru a cuantifica optimismul introdus atunci cand acelasi proces de CV este folosit atat pentru tuning, cat si pentru raportare.


In [ ]:
# GRADED FUNCTION: compare_nested_vs_non_nested
def compare_nested_vs_non_nested(model, param_grid, X, y, cv_splits=3, scoring='accuracy', random_state=42):
    """
    Compare nested CV with a non-nested tuning estimate.
    """
    ### ÎNCEPUT DE COD AICI ### (approximately 10 lines)
    comparison = None
    ### SFÂRȘIT DE COD AICI ###
    return comparison


In [ ]:
comparison = compare_nested_vs_non_nested(base_model, param_grid, X_nested, y_nested)
comparison


In [ ]:
unittests.exercise_3(run_nested_cv)
unittests.exercise_4(compare_nested_vs_non_nested)


<a name='3'></a>
## 3 - Cross-validare de la zero


<a name='ex-5'></a>
### Exercitiul 5 - `kfold_indices`

Implementeaza logica de generare a indicilor din spatele cross-validarii K-fold fara sa folosesti splittere din scikit-learn.


In [ ]:
# GRADED FUNCTION: kfold_indices
def kfold_indices(n_samples, n_splits=5, shuffle=False, random_state=None):
    """
    Build K-fold train and validation index arrays from scratch.
    """
    ### ÎNCEPUT DE COD AICI ### (approximately 18 lines)
    splits = None
    ### SFÂRȘIT DE COD AICI ###
    return splits


In [ ]:
basic_splits = kfold_indices(n_samples=20, n_splits=5, shuffle=False)
basic_splits[:2]


<a name='ex-6'></a>
### Exercitiul 6 - `stratified_kfold_indices`

Extinde acum splitterul astfel incat fold-urile de validare sa pastreze aproximativ echilibrul initial al claselor.


In [ ]:
# GRADED FUNCTION: stratified_kfold_indices
def stratified_kfold_indices(y, n_splits=5, shuffle=False, random_state=None):
    """
    Build stratified train and validation index arrays from scratch.
    """
    ### ÎNCEPUT DE COD AICI ### (approximately 24 lines)
    splits = None
    ### SFÂRȘIT DE COD AICI ###
    return splits


In [ ]:
y_demo = np.array([0] * 16 + [1] * 4)
stratified_splits = stratified_kfold_indices(y_demo, n_splits=4)
stratified_splits[:2]


In [ ]:
unittests.exercise_5(kfold_indices)
unittests.exercise_6(stratified_kfold_indices)
